# 03 - Inference with the packaged model
Loads the saved pipeline (`models/model.joblib`) and scores new patients - no manual preprocessing needed, which is the point of packaging the transformer together with the classifier.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import json
import pandas as pd
from src.predict import load_metadata, load_model, predict_records

model = load_model()
meta = load_metadata()
print("model:", meta["model_name"], "| trained:", meta["trained_at"])
print("sklearn:", meta["sklearn_version"])

model: logistic_regression | trained: 2026-10-16T08:42:59.652982+00:00
sklearn: 1.8.0


In [2]:
patients = [
    {"age": 63, "sex": 1, "cp": 1, "trestbps": 145, "chol": 233, "fbs": 1, "restecg": 2,
     "thalach": 150, "exang": 0, "oldpeak": 2.3, "slope": 3, "ca": 0, "thal": 6},
    {"age": 67, "sex": 1, "cp": 4, "trestbps": 160, "chol": 286, "fbs": 0, "restecg": 2,
     "thalach": 108, "exang": 1, "oldpeak": 1.5, "slope": 2, "ca": 3, "thal": 3},
    {"age": 41, "sex": 0, "cp": 2, "trestbps": 130, "chol": 204, "fbs": 0, "restecg": 2,
     "thalach": 172, "exang": 0, "oldpeak": 1.4, "slope": 1, "ca": 0, "thal": 3},
]
pd.DataFrame(predict_records(model, patients))

,prediction,label,confidence,probability_disease
0,0,no heart disease likely,0.8275,0.1725
1,1,heart disease likely,0.9835,0.9835
2,0,no heart disease likely,0.9616,0.0384


## Same request through the API
```bash
curl -X POST http://localhost:8000/predict -H 'Content-Type: application/json' \
  -d '{"age":67,"sex":1,"cp":4,"trestbps":160,"chol":286,"fbs":0,"restecg":2,"thalach":108,"exang":1,"oldpeak":1.5,"slope":2,"ca":3,"thal":3}'
```

## Load the same model through MLflow
(works after `python -m src.train` has created `mlflow.db`)

In [3]:
import mlflow
mlflow.set_tracking_uri(f"sqlite:///{ROOT / 'mlflow.db'}")
loaded = mlflow.sklearn.load_model(f"models:/{'heart-disease-classifier'}/1")
loaded.predict_proba(pd.DataFrame(patients)[model.feature_names_in_])[:, 1].round(3)

array([0.173, 0.983, 0.038])